# 10_v5c_tlt_dbmf_test — 对冲层重构

> 重新设计对冲层: 黄金减半 + 去 BCX + 加 TLT + 加 DBMF

## 配置对比

```
V5C 3.1 (基准):  GLDM 20 / BCX 10                     (对冲 30%)
V5C 3.1e (新):   GLDM 10 / TLT 10 / DBMF 10            (对冲 30%)
```

对冲层合计仍 30%, 但**重构为三类危机 hedge 平均分布**:
- **GLDM 10%**: 地缘黑天鹅 + 大多数危机
- **TLT 10%**:  通缩型危机 (2008 GFC, 2020 COVID)
- **DBMF 10%**: 滞胀/趋势型危机 (2022 Bear)
- **BCX 0%**:   已发现 BCX 与 VOO 相关性 0.333, 不是真 hedge, 完全去除

## 三个测试 (因 DBMF 数据限制)

| 测试 | 时间窗口 | DBMF 来源 | 配置 |
|---|---|---|---|
| **A** | 2002-2026 (23.8Y) | 无 (DBMF 数据起 2019) | GLDM 15 / TLT 15 (DBMF 10% 平分给 GLDM/TLT) |
| **B** | 2010-2026 (16Y)   | AQRIX 代理 | GLDM 10 / TLT 10 / AQRIX 10 |
| **C** | 2019-2026 (7Y)    | 实际 DBMF | GLDM 10 / TLT 10 / DBMF 10 (真实方案) |

测试 B 是核心验证 (覆盖 2018, 2020, 2022 三个关键时期).

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

tickers = ['VFINX','QQQ','HQH','XLV','VFITX','GLD','GC=F','DBC','PCRIX','TLT','DBMF','AQRIX']
raw = yf.download(tickers, start='1999-01-01', auto_adjust=True)['Close']

print('数据起始日期:')
for t in tickers:
    if t in raw.columns:
        first = raw[t].first_valid_index()
        n = raw[t].notna().sum()
        print(f'  {t:<8}: {first.date() if first else "N/A":<12}  {n} 天')

In [ ]:
def synthesize(short, long):
    short = short.dropna(); long = long.dropna()
    if len(short) == 0: return long
    overlap = short.index[0]
    if overlap <= long.index[0]: return short
    long_at = long.loc[:overlap].iloc[-1] if not long.loc[:overlap].empty else long.iloc[0]
    scale = short.iloc[0] / long_at
    early = long.loc[:overlap].iloc[:-1] * scale
    return pd.concat([early, short]).sort_index().pipe(lambda s: s[~s.index.duplicated(keep='last')])

data = pd.DataFrame({
    'VOO': raw['VFINX'], 'QQQ': raw['QQQ'],
    'HQH': raw['HQH'], 'XLV': raw['XLV'],
    'VGSH': raw['VFITX'],
    'GLDM': synthesize(raw['GLD'], raw['GC=F']),
    'BCX':  synthesize(raw['DBC'], raw['PCRIX']),
    'TLT':  raw['TLT'],
    'DBMF_proxy': raw['AQRIX'],
    'DBMF_actual': raw['DBMF'],
})

for c in data.columns:
    fv = data[c].first_valid_index()
    print(f'  {c:<14}: {fv.date() if fv else "N/A"}')

In [ ]:
def simulate_rebalance(returns_df, target_weights, threshold_pp=5.0):
    used = [t for t in target_weights.keys() if t in returns_df.columns]
    sub = returns_df[used].dropna()
    target = np.array([target_weights[t] for t in used]); target = target/target.sum()
    cw = target.copy(); pr=[]; rd=[sub.index[0]]
    for date, dr in sub.iterrows():
        pr.append(np.sum(cw*dr.values))
        nw = cw*(1+dr.values); nw = nw/nw.sum()
        if np.max(np.abs(nw-target))*100 >= threshold_pp:
            cw = target.copy(); rd.append(date)
        else:
            cw = nw
    return pd.Series(pr, index=sub.index), rd

def metrics(rs, dates, name):
    cum = (1+rs).cumprod()
    n_y = len(rs)/252
    cagr = cum.iloc[-1]**(1/n_y) - 1
    vol = rs.std()*np.sqrt(252)
    sharpe = (cagr-0.04)/vol
    downside = rs[rs<0]
    sortino = (cagr-0.04)/(downside.std()*np.sqrt(252))
    rm = cum.expanding().max()
    dd = (cum/rm - 1)
    return {'Name':name,'CAGR':cagr,'Vol':vol,'Sharpe':sharpe,'Sortino':sortino,
            'Max DD':dd.min(),'Max DD Date':dd.idxmin(),
            'Calmar':cagr/abs(dd.min()),'Rebalances':len(dates)-1,'Years':n_y}

def show_compare(metric_list):
    cols = ['CAGR','Vol','Sharpe','Sortino','Max DD','Calmar']
    print(f"{'Metric':<10}", end='')
    for m in metric_list: print(f"  {m['Name']:<25}", end='')
    print()
    print('-'*100)
    for col in cols:
        fmt = '{:.2%}' if col not in ['Sharpe','Sortino','Calmar'] else '{:.3f}'
        line = f'{col:<10}'
        for m in metric_list:
            line += f'  {fmt.format(m[col]):<25}'
        print(line)
    print(f"{'Rebalances':<10}", end='')
    for m in metric_list: print(f"  {str(m['Rebalances']):<25}", end='')
    print()

In [ ]:
# ============================================================
# 测试 A: 23.8Y - 无 DBMF, GLDM 15 / TLT 15 (DBMF 10% 平分)
# ============================================================
print('=' * 90)
print('测试 A: 23.8Y - 无 DBMF (GLDM 15 / TLT 15, DBMF 10% 平分给 GLDM 和 TLT)')
print('=' * 90)

data_A = data[['VOO','QQQ','HQH','XLV','VGSH','GLDM','BCX','TLT']].dropna()
ret_A = data_A.pct_change().dropna()
print(f'窗口: {data_A.index[0].date()} → {data_A.index[-1].date()} ({len(data_A)/252:.1f} 年)')

V5C_3_1 = {'VOO':0.15,'QQQ':0.15,'HQH':0.10,'XLV':0.10,'GLDM':0.20,'BCX':0.10,'VGSH':0.20}
# Test A: 没 DBMF, 把它的 10% 平分给 GLDM 和 TLT
V5C_3_1e_A = {'VOO':0.15,'QQQ':0.15,'HQH':0.10,'XLV':0.10,'GLDM':0.15,'TLT':0.15,'VGSH':0.20}

r_A_b, d_A_b = simulate_rebalance(ret_A, V5C_3_1)
r_A_e, d_A_e = simulate_rebalance(ret_A, V5C_3_1e_A)

show_compare([
    metrics(r_A_b, d_A_b, '3.1 (GLDM20+BCX10)'),
    metrics(r_A_e, d_A_e, '3.1e-A (GLDM15+TLT15)'),
])

print('\n关键时期表现:')
events_A = {
    '2008 GFC':       ('2007-10-09', '2009-03-09'),
    '2008 急跌':       ('2008-09-01', '2008-12-31'),
    '2018-Q4':         ('2018-10-01', '2018-12-31'),
    '2020 COVID':     ('2020-02-19', '2020-04-30'),
    '2022 Bear':      ('2022-01-01', '2022-12-31'),
    '2025 Q1 关税':    ('2025-01-01', '2025-04-30'),
}
print(f"{'时期':<22} {'3.1':>10} {'3.1e-A':>10} {'Δ':>10}")
for n, (s, e) in events_A.items():
    if pd.Timestamp(s) < r_A_b.index[0]: continue
    a = (1 + r_A_b.loc[s:e]).prod() - 1
    b = (1 + r_A_e.loc[s:e]).prod() - 1
    print(f'{n:<22} {a:>+9.2%}  {b:>+9.2%}  {b-a:>+9.2%}')

In [ ]:
# ============================================================
# 测试 B (核心): 16Y - GLDM 10 + TLT 10 + AQRIX 10 (BCX 0)
# ============================================================
print('=' * 90)
print('测试 B: 16Y - GLDM 10 / TLT 10 / AQRIX 10 (BCX 完全去除)')
print('=' * 90)

data_B = data[['VOO','QQQ','HQH','XLV','VGSH','GLDM','BCX','TLT','DBMF_proxy']].dropna()
ret_B = data_B.pct_change().dropna()
print(f'窗口: {data_B.index[0].date()} → {data_B.index[-1].date()} ({len(data_B)/252:.1f} 年)')

V5C_3_1e = {'VOO':0.15,'QQQ':0.15,'HQH':0.10,'XLV':0.10,
             'GLDM':0.10,'TLT':0.10,'DBMF_proxy':0.10,'VGSH':0.20}

r_B_b, d_B_b = simulate_rebalance(ret_B, V5C_3_1)
r_B_e, d_B_e = simulate_rebalance(ret_B, V5C_3_1e)

show_compare([
    metrics(r_B_b, d_B_b, '3.1 (基准)'),
    metrics(r_B_e, d_B_e, '3.1e (G10+T10+D10)'),
])

print('\n关键时期 (16Y 窗口):')
events_B = {
    '2018-Q4 跌势':      ('2018-10-01', '2018-12-31'),
    '2020 COVID':        ('2020-02-19', '2020-04-30'),
    '2020 流动性极端':    ('2020-03-09', '2020-03-23'),
    '2022 Bear (全年)':   ('2022-01-01', '2022-12-31'),
    '2022 滞胀核心':      ('2022-01-01', '2022-09-30'),
    '2023-2024 复苏':    ('2023-01-01', '2024-12-31'),
    '2025 Q1 关税':       ('2025-01-01', '2025-04-30'),
}
print(f"{'时期':<22} {'3.1':>10} {'3.1e':>10} {'Δ':>10}")
for n, (s, e) in events_B.items():
    a = (1 + r_B_b.loc[s:e]).prod() - 1
    b = (1 + r_B_e.loc[s:e]).prod() - 1
    print(f'{n:<22} {a:>+9.2%} {b:>+9.2%} {b-a:>+9.2%}')

In [ ]:
# ============================================================
# 测试 C: 7Y - 实际 DBMF (验证 AQRIX 代理可信度)
# ============================================================
print('=' * 90)
print('测试 C: 7Y - 实际 DBMF + AQRIX 对照')
print('=' * 90)

data_C = data[['VOO','QQQ','HQH','XLV','VGSH','GLDM','BCX','TLT','DBMF_actual','DBMF_proxy']].dropna()
ret_C = data_C.pct_change().dropna()
print(f'窗口: {data_C.index[0].date()} → {data_C.index[-1].date()} ({len(data_C)/252:.1f} 年)')

V5C_3_1e_actual = {'VOO':0.15,'QQQ':0.15,'HQH':0.10,'XLV':0.10,
                    'GLDM':0.10,'TLT':0.10,'DBMF_actual':0.10,'VGSH':0.20}
V5C_3_1e_proxy = {'VOO':0.15,'QQQ':0.15,'HQH':0.10,'XLV':0.10,
                   'GLDM':0.10,'TLT':0.10,'DBMF_proxy':0.10,'VGSH':0.20}

r_C_b, d_C_b = simulate_rebalance(ret_C, V5C_3_1)
r_C_a, d_C_a = simulate_rebalance(ret_C, V5C_3_1e_actual)
r_C_p, d_C_p = simulate_rebalance(ret_C, V5C_3_1e_proxy)

show_compare([
    metrics(r_C_b, d_C_b, '3.1 (基准)'),
    metrics(r_C_a, d_C_a, '3.1e (实际 DBMF)'),
    metrics(r_C_p, d_C_p, '3.1e (AQRIX 代理)'),
])

print('\nDBMF vs AQRIX 单标的对比 (7Y):')
for label, ser in [('DBMF 实际', ret_C['DBMF_actual']), ('AQRIX 代理', ret_C['DBMF_proxy'])]:
    cum = (1+ser).cumprod()
    cagr = cum.iloc[-1]**(252/len(ser))-1
    vol = ser.std()*np.sqrt(252)
    sharpe = (cagr-0.04)/vol
    rm = cum.expanding().max()
    dd = (cum/rm - 1).min()
    print(f'  {label:<10}: CAGR {cagr:>+6.2%}  Sharpe {sharpe:>+.3f}  MaxDD {dd:>+6.2%}')
corr = ret_C['DBMF_actual'].corr(ret_C['DBMF_proxy'])
print(f'\n相关性 DBMF vs AQRIX: {corr:.3f}  (>0.7 表示代理可信)')

In [ ]:
# ============================================================
# 单标的特征对比 (TLT vs GLDM vs BCX vs DBMF/AQRIX)
# ============================================================
print('=' * 90)
print('单标的特征对比 (23.8Y for VOO/GLDM/BCX/TLT/VGSH; 16Y for AQRIX; 7Y for DBMF)')
print('=' * 90)

for label in ['VOO','GLDM','BCX','TLT','VGSH']:
    if label in ret_A.columns:
        s = ret_A[label]
        cum = (1+s).cumprod(); n = len(s)
        cagr = cum.iloc[-1]**(252/n)-1
        vol = s.std()*np.sqrt(252)
        sharpe = (cagr-0.04)/vol
        rm = cum.expanding().max()
        dd = (cum/rm - 1).min()
        cv = s.corr(ret_A['VOO'])
        print(f'{label:<6}: CAGR {cagr:>+6.2%}  Sharpe {sharpe:>+.3f}  MaxDD {dd:>+7.2%}  vsVOO {cv:>+.3f}')

for label in ['DBMF_proxy']:
    s = ret_B[label]
    cum = (1+s).cumprod(); n = len(s)
    cagr = cum.iloc[-1]**(252/n)-1
    vol = s.std()*np.sqrt(252)
    sharpe = (cagr-0.04)/vol
    rm = cum.expanding().max()
    dd = (cum/rm - 1).min()
    cv = s.corr(ret_B['VOO'])
    print(f'AQRIX : CAGR {cagr:>+6.2%}  Sharpe {sharpe:>+.3f}  MaxDD {dd:>+7.2%}  vsVOO {cv:>+.3f}  (16Y)')

print('\n危机期单标的表现 (23.8Y 窗口):')
print(f"{'时期':<22} {'VOO':>8} {'GLDM':>8} {'BCX':>8} {'TLT':>8} {'VGSH':>8}")
for n, (s_d, e_d) in events_A.items():
    if pd.Timestamp(s_d) < ret_A.index[0]: continue
    row = f'{n:<22}'
    for label in ['VOO','GLDM','BCX','TLT','VGSH']:
        r = (1 + ret_A[label].loc[s_d:e_d]).prod() - 1
        row += f' {r:>+7.2%}'
    print(row)

In [ ]:
# ============================================================
# 测试 B 净值 + 回撤
# ============================================================
cum_b = (1 + r_B_b).cumprod()
cum_e = (1 + r_B_e).cumprod()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
axes[0].plot(cum_b, label='V5C 3.1 (基准: GLDM20+BCX10)', linewidth=2, alpha=0.85)
axes[0].plot(cum_e, label='V5C 3.1e (GLDM10+TLT10+AQRIX10)', linewidth=2, alpha=0.85)
axes[0].set_title('测试 B 净值 (16Y, log scale)', fontsize=14)
axes[0].set_yscale('log')
axes[0].legend()
axes[0].grid(alpha=0.3)

for cum, label, color in [(cum_b, '3.1', 'C0'), (cum_e, '3.1e', 'C2')]:
    rm = cum.expanding().max()
    dd = (cum / rm) - 1
    axes[1].fill_between(dd.index, dd.values, 0, alpha=0.4, color=color, label=label)
axes[1].set_title('回撤对比', fontsize=14)
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 解读思路

### 测试 A (23.8Y, 无 DBMF, GLDM15+TLT15)
- **关注 2008 GFC**: TLT 在 2008 +33%, GLDM 仅 +5%, 所以 3.1e-A 应当显著优于 3.1
- **关注 2022 Bear**: TLT 在 2022 -32%, 没有 DBMF 补救, 3.1e-A 应当显著差于 3.1
- **整体 Sharpe**: 看哪边更优

### 测试 B (16Y, AQRIX 代理, 完整方案)
- **关注 2018-Q4 + 2020 COVID**: TLT + GLDM 均应受益
- **关注 2022 Bear**: AQRIX 应当 +20%+, 抵消 TLT 的 -32%, 3.1e 应当不差于 3.1
- **整体 Sharpe**: 这是核心判断 — Sharpe 是否超越 3.1?

### 测试 C (7Y)
- 验证 AQRIX 与实际 DBMF 是否一致
- 相关性 > 0.7 表示代理可信

## 决策标准

**采纳 V5C 3.1e (作为 V5C 4.0 候选)** 仅当:
1. 测试 B Sharpe ≥ V5C 3.1 (含 BCX) 的 Sharpe
2. 2022 Bear 表现 不差于 3.1 超过 3pp
3. 测试 C 中 AQRIX vs 实际 DBMF 相关性 > 0.7

**保持 V5C 3.1** 如果:
- TLT 2022 大跌拖累 Sharpe 下降 > 0.05
- DBMF 数据期不足 6 年的根本顾虑未解 (历史是否会有 chop 期失效?)

## 重要提醒

**本测试是 V5C 4.0 (2028+) 的研究输入, 不修改 V5C 3.1 (24 个月承诺仍有效)**.
结果记录到 `reflections/04-V5C-3.1-research.md` 末尾, 作为 2028 设计参考.